# Market Relevance Scoring

This notebook scores market relevance from `full_data_to_2040`. Run the modeling notebook first so `full_data_to_2040` exists in memory.

## 1. Set Up

In [ ]:
%reload_ext autoreload
%autoreload 2

%cd /Users/dunglai/Documents/Việt Dũng/Personal Projects/World Foresight Framework/Code

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)

from Scoring.scoring import score_question

PROJECT_ROOT = Path("/Users/dunglai/Documents/Việt Dũng/Personal Projects/World Foresight Framework")


## 2. Prepare Scoring Input

In [ ]:
full_data_to_2040 = pd.read_excel(
    PROJECT_ROOT / "Final Data.xlsx",
    sheet_name="Final Full Data",
)
full_data_to_2040.head(5)


In [ ]:
required_columns = ["id", "market", "date", "value", "scenario"]
missing_columns = [col for col in required_columns if col not in full_data_to_2040.columns]
if missing_columns:
    raise ValueError(f"full_data_to_2040 is missing columns: {missing_columns}")

scoring_input_df = full_data_to_2040[full_data_to_2040["scenario"] == "main_scenario"].copy()
scoring_input_df["date"] = scoring_input_df["date"].astype(int)

available_proxy_ids = sorted(scoring_input_df["id"].dropna().astype(str).unique())
print("Scoring rows:", len(scoring_input_df))
print("Available proxy ids:", available_proxy_ids)
display(scoring_input_df.head())

## 3. Confirm Proxy Scoring Config

Edit this config before scoring. Direction must be analyst-confirmed for every proxy: `positive` means higher value is better; `negative` means higher value is worse.

In [ ]:
QUESTION_NAME = "Q1"

proxy_scoring_config = pd.DataFrame([
    {"id": "D1", "question": QUESTION_NAME, "direction": "positive"},
    {"id": "D2", "question": QUESTION_NAME, "direction": "positive"},
    {"id": "D3", "question": QUESTION_NAME, "direction": "positive"},
    {"id": "D4", "question": QUESTION_NAME, "direction": "positive"},
])

display(proxy_scoring_config)

## 4. Score Market Relevance

In [ ]:
TARGET_YEAR = 2040

market_relevance_scores = score_question(
    scoring_input_df,
    proxy_scoring_config,
    question=QUESTION_NAME,
    target_year=TARGET_YEAR,
)

print("Score rows:", len(market_relevance_scores))
display(market_relevance_scores.sort_values(["rank"]).head(30))